## **ARTICULOS**

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
from pyspark.sql.types import *
articulo = spark.read.table("lh_retailnova_bronze_dev.articulos")
display(articulo)

StatementMeta(, 43f90a1d-26d9-474b-bc8f-346602248b41, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 414c7245-6d14-4a22-9497-6a0a9e4f8d92)

## **MIEMBROS**

In [3]:
from pyspark.sql.types import *
miembros = spark.read.table("lh_retailnova_bronze_dev.miembros")
display(miembros)

StatementMeta(, 43f90a1d-26d9-474b-bc8f-346602248b41, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2d30a256-b1e4-4cd6-b9ce-10b2d49ca551)

## **DEVOLUCIÓN**

In [4]:
from pyspark.sql.types import *
devolucion = spark.read.table("lh_retailnova_bronze_dev.devolucion")
display(devolucion)

StatementMeta(, 43f90a1d-26d9-474b-bc8f-346602248b41, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ea930c9b-502f-4fd3-a9ba-fbdf69ccf1f9)

## **PROVEEDORES**

In [5]:
from pyspark.sql.types import *
proveedor = spark.read.table("lh_retailnova_bronze_dev.proveedores")
display(proveedor)

StatementMeta(, 43f90a1d-26d9-474b-bc8f-346602248b41, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a588efd5-0363-4689-8fe2-d7e93ee9d7a2)

## **STOCK**

In [6]:
from pyspark.sql.types import *
stock = spark.read.table("lh_retailnova_bronze_dev.stock")
display(stock)

StatementMeta(, 43f90a1d-26d9-474b-bc8f-346602248b41, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5525128c-228b-474e-afb4-95a362488220)

## **TIENDA**

In [7]:
from pyspark.sql.types import *
tienda = spark.read.table("lh_retailnova_bronze_dev.tienda")
display(tienda)

StatementMeta(, 43f90a1d-26d9-474b-bc8f-346602248b41, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7b35d003-a41e-4587-ab84-815a6a8192d3)

In [8]:
from pyspark.sql.types import *
venta = spark.read.table("lh_retailnova_bronze_dev.venta")
display(venta)

StatementMeta(, 43f90a1d-26d9-474b-bc8f-346602248b41, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4119fbcf-dafd-4fbe-b3b8-3c6bb157d39f)

In [9]:
from pyspark.sql import functions as F
from datetime import datetime
import time

# 1. Definir los valores de auditoría para esta ejecución
inicio_tiempo = time.time()
timestamp_ingesta = datetime.utcnow()
id_lote_procesamiento = f"lote_{int(inicio_tiempo)}"
sistema_fuente = "sql_origen_retailnova"

# 2. Leer tu tabla actual del Lakehouse
df = spark.read.table("lh_retailnova_bronze_dev.venta")

# 3. Agregar las columnas de metadatos y las de partición (Año, Mes, Día)
df_con_auditoria = (df
    .withColumn("meta_fecha_ingesta", F.lit(timestamp_ingesta))
    .withColumn("meta_sistema_fuente", F.lit(sistema_fuente))
    .withColumn("meta_id_lote", F.lit(id_lote_procesamiento))
    # Creamos las columnas para particionar físicamente por fecha si tu tabla tiene una fecha (ej. fecha_trans)
    .withColumn("anio_ingesta", F.year(F.col("fecha_trans")))
    .withColumn("mes_ingesta", F.month(F.col("fecha_trans")))
    .withColumn("dia_ingesta", F.dayofmonth(F.col("fecha_trans")))
)

display(df_con_auditoria.limit(5))

StatementMeta(, 43f90a1d-26d9-474b-bc8f-346602248b41, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5b14ddeb-04bf-4151-aa5d-57a53a9696c5)

In [10]:
# Calcular métricas finales para el Log
fin_tiempo = time.time()
duracion_total = round(fin_tiempo - inicio_tiempo, 2)
registros_procesados = df_con_auditoria.count()

# Obtener tamaño aproximado en disco de la tabla Delta (en Bytes o MB)
tamanio_tabla_bytes = spark.sql("DESCRIBE DETAIL lh_retailnova_bronze_dev.venta").select("sizeInBytes").collect()[0]["sizeInBytes"]
tamanio_mb = round(tamanio_tabla_bytes / (1024 * 1024), 2)

# Imprimir el reporte de Log oficial de la ejecución
print("==================================================")
print("             LOG DE EJECUCIÓN - BRONZE           ")
print("==================================================")
print(f"ID de Lote          : {id_lote_procesamiento}")
print(f"Fecha y Hora        : {timestamp_ingesta}")
print(f"Registros Procesados: {registros_procesados:,}") 
print(f"Tamaño en Disco     : {tamanio_mb} MB")
print(f"Duración del Proceso: {duracion_total} segundos")
print("==================================================")

StatementMeta(, 43f90a1d-26d9-474b-bc8f-346602248b41, 12, Finished, Available, Finished, False)

             LOG DE EJECUCIÓN - BRONZE           
ID de Lote          : lote_1785337280
Fecha y Hora        : 2026-07-29 15:01:20.021427
Registros Procesados: 1,000,000
Tamaño en Disco     : 27.72 MB
Duración del Proceso: 2.08 segundos
